# Initial Review of the Product List
The first step is to review the product list at a high level.  
The goal is to understand what types of products appear in the dataset, how product names are written,  
and what kinds of inconsistencies exist across the listings.

# Defining Duplicates
The next step is to define what should be considered a duplicate product.  
At this stage, the goal is to distinguish between listings that refer to the same product  
and listings that may look similar but actually describe different variants or different items.

# Assumptions
For the purpose of this solution, I assume that the input includes at least a product title field and a price field.  
The title field is used for normalization and duplicate matching, while the price field is used to select the lowest-priced listing for customer display.

# Initial Normalization
The next step is to normalize product titles in order to reduce superficial differences between listings.  
This includes actions such as lowercasing, trimming extra spaces, removing unnecessary punctuation, and standardizing common text patterns.  
The goal is to create a cleaner and more consistent representation before duplicate detection.

In [ ]:
import pandas as pd
import re

def normalize_product_title(title: str) -> str:
    if pd.isna(title):
        return "" # return empty string for NaN values
    
    title = title.lower() # convert to lowercase
    title = re.sub(r'[^\w\s]', '', title) # remove punctuation
    title = re.sub(r'\s+', ' ', title).strip() # remove extra whitespace
    return title

products_df = pd.DataFrame(products_list, columns=['product_title']) # create DataFrame from the products_list
products_df['normalized_title'] = products_df['product_title'].apply(normalize_product_title) # apply normalization function to the product_title column

# Synonym and Language Mapping
After the initial normalization step, the next stage is to map different language variants and synonymous product terms into a more consistent representation.  
This helps connect listings that refer to the same product using different languages, spellings, or naming conventions.  
This mapping is not intended to manually cover every product in the dataset.  
Instead, it serves as an initial curated layer that can later be expanded based on repeated patterns found in the product list.

In [ ]:
brand_mapping = {
    "apple": ["Apple", "אפל"], 
    "samsung": ["Samsung", "סמסונג"],
    "sony": ["Sony", "סוני"]
}

family_mapping = {
    "iphone": ["iPhone", "אייפון"],
    "galaxy": ["Galaxy", "גלקסי"],
    "xperia": ["Xperia", "אקספריה"]
}

variant_normalization_mapping = {
    "pro": ["Pro", "פרו"],
    "max": ["Max", "מקס"],
    "ultra": ["Ultra", "אולטרה"],
    "plus": ["Plus", "פלוס"]
}

term_normalization_mapping = {
    "128 gb": ["128 gb", "128GB", "128 גיגה"],
    "256 gb": ["256 gb", "256GB", "256 גיגה"],
    "512 gb": ["512 gb", "512GB", "512 גיגה"],
    "s 23": ["S23", "S 23", "S-23"]
}

# Applying the Mapping Layer
At this stage, the mapping rules are applied to product titles in order to replace language variants, naming differences, and common synonyms with a more consistent representation.  
The goal is to make similar listings easier to compare before applying duplicate matching rules.

In [ ]:
def apply_mapping(title: str, mapping_dict: dict) -> str:
    for standard_term, variants in mapping_dict.items():
        for variant in variants:
            if variant.lower() in title:
                title = title.replace(variant.lower(), standard_term)
    return title

def apply_all_mappings(title: str) -> str:
    title = apply_mapping(title, brand_mapping)
    title = apply_mapping(title, family_mapping)
    title = apply_mapping(title, variant_normalization_mapping)
    title = apply_mapping(title, term_normalization_mapping)
    return title

products_df['mapped_title'] = products_df['normalized_title'].apply(apply_all_mappings) # apply all mappings to the normalized_title column

# Exact Duplicate Detection
At this stage, listings with the same mapped title can be treated as exact duplicates.  
These records are grouped together, and the listing with the lowest price is selected for customer display.

In [ ]:
lowest_price_products = products_df.loc[products_df.groupby('mapped_title')['price'].idxmin()] # keep the row with the lowest price for each unique mapped title

# Summary
This approach combines text normalization, synonym and language mapping, and exact duplicate detection in order to unify inconsistent product listings.  
After grouping equivalent records, the lowest-priced listing is selected for customer display.  
This creates a simple and explainable baseline that can later be extended with more advanced matching logic if needed.